# 02 — Train Aesthetic Ranker + Style Classifier

Uses the frames and labels produced by **01_data_collection.ipynb**.

### What gets trained

| Model | Task | Output |
|-------|------|--------|
| `AestheticRanker` | Regression — predict `aesthetic_score` (0–1) | `aesthetic_ranker.pt` |
| `StyleClassifier` | 8-class classification — predict video style | `style_classifier.pt` |

Both models share the same frozen **CLIP ViT-B-32** backbone. Only the lightweight MLP heads are trained.

### 8 Style Classes
```
0 = anime          4 = gaming
1 = movie_villain  5 = athlete
2 = rap_hiphop     6 = ufc_mma
3 = motivational   7 = aesthetic_flow
```

Trained models + class index map are saved to `MyDrive/training_data/weights/`.

In [ ]:
!pip install -q open-clip-torch torch torchvision pandas scikit-learn tqdm matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Paths (must match Notebook 1) ──
LABELS_PATH  = '/content/drive/MyDrive/training_data/labels.csv'
EMBED_PATH   = '/content/drive/MyDrive/training_data/embeddings.pt'
WEIGHTS_DIR  = '/content/drive/MyDrive/training_data/weights'

import os
os.makedirs(WEIGHTS_DIR, exist_ok=True)

# ── Training settings ──
BATCH_SIZE      = 256
EPOCHS_RANKER   = 30    # aesthetic regression
EPOCHS_STYLE    = 40    # style classification
LR              = 3e-4
DROPOUT         = 0.3
VAL_SPLIT       = 0.1   # 10% held out for validation
SEED            = 42

# ── Category definitions (must stay in this order) ──
CATEGORY_ORDER = [
    'anime', 'movie_villain', 'rap_hiphop', 'motivational',
    'gaming', 'athlete', 'ufc_mma', 'aesthetic_flow'
]
cat_to_idx = {c: i for i, c in enumerate(CATEGORY_ORDER)}
NUM_CLASSES = len(CATEGORY_ORDER)

print(f"Training {NUM_CLASSES} style classes: {CATEGORY_ORDER}")

In [ ]:
import pandas as pd
import torch
import numpy as np

# Load labels
df = pd.read_csv(LABELS_PATH)
df = df[df['category_idx'] >= 0].reset_index(drop=True)  # drop rows with unknown category
print(f"Total labelled frames: {len(df)}")
print(df['category'].value_counts().to_string())

# Load precomputed CLIP embeddings
saved      = torch.load(EMBED_PATH, map_location='cpu')
embed_dict = dict(zip(saved['frame_names'], range(len(saved['frame_names']))))
embeddings = saved['embeddings']  # (N, 512)

# Align: only keep frames that exist in both labels and embeddings
df = df[df['frame_path'].isin(embed_dict)].reset_index(drop=True)
print(f"\nFrames with embeddings: {len(df)}")

# Build aligned tensors
idxs   = [embed_dict[p] for p in df['frame_path']]
X      = embeddings[idxs].float()          # (M, 512) — CLIP features
y_aes  = torch.tensor(df['aesthetic_score'].values, dtype=torch.float32)  # regression targets
y_cat  = torch.tensor(df['category_idx'].values,    dtype=torch.long)     # class labels
print(f"\nX shape : {X.shape}")
print(f"y_aes   : min={y_aes.min():.3f}  max={y_aes.max():.3f}  mean={y_aes.mean():.3f}")
print(f"y_cat   : classes present = {torch.unique(y_cat).tolist()}")

In [ ]:
from torch.utils.data import Dataset, DataLoader, random_split

class FrameDataset(Dataset):
    """Simple dataset that wraps precomputed CLIP embeddings."""
    def __init__(self, X, y_aes, y_cat):
        self.X     = X
        self.y_aes = y_aes
        self.y_cat = y_cat

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y_aes[i], self.y_cat[i]


torch.manual_seed(SEED)
full_dataset = FrameDataset(X, y_aes, y_cat)
val_size     = int(len(full_dataset) * VAL_SPLIT)
train_size   = len(full_dataset) - val_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {train_size}  |  Val: {val_size}")
print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")

In [ ]:
import torch.nn as nn

class AestheticRanker(nn.Module):
    """Lightweight MLP that maps a 512-d CLIP embedding → aesthetic score in [0,1]."""
    def __init__(self, emb_dim: int = 512, dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
            nn.Sigmoid(),   # output in [0, 1]
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class StyleClassifier(nn.Module):
    """Lightweight MLP that maps a 512-d CLIP embedding → 8-class style logits."""
    def __init__(self, emb_dim: int = 512, num_classes: int = 8, dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),  # raw logits → use CrossEntropyLoss
        )

    def forward(self, x):
        return self.net(x)


DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

ranker     = AestheticRanker(dropout=DROPOUT).to(DEVICE)
classifier = StyleClassifier(num_classes=NUM_CLASSES, dropout=DROPOUT).to(DEVICE)

print(f"AestheticRanker  params: {sum(p.numel() for p in ranker.parameters()):,}")
print(f"StyleClassifier  params: {sum(p.numel() for p in classifier.parameters()):,}")

In [ ]:
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

def train_model(model, loss_fn, task, epochs, loader_tr, loader_val):
    """Generic training loop for either regression or classification."""
    opt    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    hist_tr, hist_val = [], []

    for epoch in range(1, epochs + 1):
        # ── Train ──
        model.train()
        running = 0.0
        for xb, y_aes, y_cat in loader_tr:
            xb    = xb.to(DEVICE)
            y_aes = y_aes.to(DEVICE)
            y_cat = y_cat.to(DEVICE)
            target = y_aes if task == 'regression' else y_cat
            opt.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, target)
            loss.backward()
            opt.step()
            running += loss.item() * len(xb)
        sched.step()
        train_loss = running / train_size

        # ── Validate ──
        model.eval()
        running = 0.0
        with torch.no_grad():
            for xb, y_aes, y_cat in loader_val:
                xb    = xb.to(DEVICE)
                y_aes = y_aes.to(DEVICE)
                y_cat = y_cat.to(DEVICE)
                target = y_aes if task == 'regression' else y_cat
                pred = model(xb)
                running += loss_fn(pred, target).item() * len(xb)
        val_loss = running / val_size

        hist_tr.append(train_loss)
        hist_val.append(val_loss)

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{epochs}  train={train_loss:.4f}  val={val_loss:.4f}")

    # Plot
    plt.figure(figsize=(8, 3))
    plt.plot(hist_tr,  label='train')
    plt.plot(hist_val, label='val')
    plt.xlabel('epoch'); plt.ylabel('loss'); plt.title(task); plt.legend()
    plt.tight_layout(); plt.show()

    return model


# ── Train AestheticRanker (MSE regression) ──
print("=== Training AestheticRanker ===")
ranker = train_model(
    ranker,
    loss_fn = nn.MSELoss(),
    task    = 'regression',
    epochs  = EPOCHS_RANKER,
    loader_tr  = train_loader,
    loader_val = val_loader,
)

# ── Train StyleClassifier (cross-entropy classification) ──
print("\n=== Training StyleClassifier ===")
classifier = train_model(
    classifier,
    loss_fn = nn.CrossEntropyLoss(),
    task    = 'classification',
    epochs  = EPOCHS_STYLE,
    loader_tr  = train_loader,
    loader_val = val_loader,
)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Collect validation predictions
ranker.eval(); classifier.eval()
all_y_aes, all_pred_aes = [], []
all_y_cat, all_pred_cat = [], []

with torch.no_grad():
    for xb, y_aes, y_cat in val_loader:
        xb = xb.to(DEVICE)
        pred_aes = ranker(xb).cpu()
        pred_cat = classifier(xb).cpu().argmax(dim=-1)
        all_y_aes.extend(y_aes.numpy())
        all_pred_aes.extend(pred_aes.numpy())
        all_y_cat.extend(y_cat.numpy())
        all_pred_cat.extend(pred_cat.numpy())

mse  = np.mean((np.array(all_y_aes) - np.array(all_pred_aes)) ** 2)
acc  = accuracy_score(all_y_cat, all_pred_cat)
print(f"AestheticRanker  val MSE   : {mse:.4f}  (RMSE: {mse**0.5:.4f})")
print(f"StyleClassifier  val Acc   : {acc:.3%}")
print()
print(classification_report(all_y_cat, all_pred_cat, target_names=CATEGORY_ORDER))

In [ ]:
import json

# Save model weights
ranker_path     = os.path.join(WEIGHTS_DIR, 'aesthetic_ranker.pt')
classifier_path = os.path.join(WEIGHTS_DIR, 'style_classifier.pt')
cat_map_path    = os.path.join(WEIGHTS_DIR, 'category_index.json')

torch.save(ranker.state_dict(),     ranker_path)
torch.save(classifier.state_dict(), classifier_path)

with open(cat_map_path, 'w') as f:
    json.dump({'idx_to_category': CATEGORY_ORDER,
               'category_to_idx': cat_to_idx}, f, indent=2)

print(f"Saved: {ranker_path}")
print(f"Saved: {classifier_path}")
print(f"Saved: {cat_map_path}")

In [ ]:
# ── Quick sanity check: score a few random val frames ──
ranker.eval(); classifier.eval()

sample_indices = torch.randint(0, len(val_ds), (5,))
with torch.no_grad():
    for i in sample_indices:
        xb, y_aes, y_cat = val_ds[i]
        pred_aes = ranker(xb.unsqueeze(0).to(DEVICE)).item()
        pred_cat = classifier(xb.unsqueeze(0).to(DEVICE)).argmax(dim=-1).item()
        true_cat = CATEGORY_ORDER[y_cat.item()]
        pred_cat_name = CATEGORY_ORDER[pred_cat]
        print(f"true_aes={y_aes.item():.3f}  pred_aes={pred_aes:.3f}  "
              f"true_style={true_cat:15s}  pred_style={pred_cat_name}")

In [ ]:
# ── TorchScript export for Mac Mini M4 inference ──
ranker.eval(); classifier.eval()

example = torch.zeros(1, 512).to(DEVICE)  # dummy input

ts_ranker     = torch.jit.trace(ranker,     example)
ts_classifier = torch.jit.trace(classifier, example)

ts_ranker_path     = os.path.join(WEIGHTS_DIR, 'aesthetic_ranker_ts.pt')
ts_classifier_path = os.path.join(WEIGHTS_DIR, 'style_classifier_ts.pt')

ts_ranker.save(ts_ranker_path)
ts_classifier.save(ts_classifier_path)

print(f"TorchScript exported:")
print(f"  {ts_ranker_path}")
print(f"  {ts_classifier_path}")
print("\nThese files can be loaded on Mac Mini M4 with torch.jit.load() — no model code needed.")

In [ ]:
# ── Copy weights to repo weights/ folder for direct use ──
import shutil

REPO_WEIGHTS_DIR = '/content/drive/MyDrive/video_ai/weights'
os.makedirs(REPO_WEIGHTS_DIR, exist_ok=True)

for fname in ['aesthetic_ranker_ts.pt', 'style_classifier_ts.pt', 'category_index.json']:
    src = os.path.join(WEIGHTS_DIR, fname)
    dst = os.path.join(REPO_WEIGHTS_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"  Copied → {dst}")

print("\nDone. These files go into your repo's video_ai/weights/ folder.")

In [ ]:
# ── Final summary ──
print("=" * 50)
print("Training complete!")
print("=" * 50)
print(f"\nModel weights saved to: {WEIGHTS_DIR}")
print(f"\nFiles generated:")
for fname in os.listdir(WEIGHTS_DIR):
    fsize = os.path.getsize(os.path.join(WEIGHTS_DIR, fname)) / 1024
    print(f"  {fname:40s} {fsize:8.1f} KB")
print("\nNext: copy weights/ folder contents to your Mac Mini and run the pipeline.")

## ✅ Training Complete

### Files in `MyDrive/training_data/weights/`

| File | Purpose |
|------|---------|
| `aesthetic_ranker.pt` | Raw state dict — aesthetic regression head |
| `style_classifier.pt` | Raw state dict — 8-class style head |
| `aesthetic_ranker_ts.pt` | TorchScript — use this on Mac Mini |
| `style_classifier_ts.pt` | TorchScript — use this on Mac Mini |
| `category_index.json` | Maps class index ↔ category name |

### How to use on Mac Mini

```python
import torch, json

ranker     = torch.jit.load('weights/aesthetic_ranker_ts.pt').eval()
classifier = torch.jit.load('weights/style_classifier_ts.pt').eval()

with open('weights/category_index.json') as f:
    idx_to_cat = json.load(f)['idx_to_category']

# embedding: torch.Tensor shape (1, 512) — from CLIP
score     = ranker(embedding).item()           # 0.0–1.0
style_idx = classifier(embedding).argmax(-1).item()
style     = idx_to_cat[style_idx]
```

**→ Proceed to the Mac Mini setup guide (`DEPLOY.md`)** to install the pipeline and run your first video.